In [55]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score

In [56]:
matches = pd.read_csv("matches.csv", index_col=0)

In [57]:
#Convert non-numerical data to numerical data to be used by the model
matches["Date"] = pd.to_datetime(matches["Date"])
matches["Home_or_Away_numeric"] = matches["Home_or_Away"].astype("category").cat.codes
matches["Team_numeric"] = matches["Team"].astype("category").cat.codes
matches["Opponent_numeric"] = matches["Opponent"].astype("category").cat.codes
matches["Target"] = (matches["Win_Loss"] == "W").astype(int)
matches

,Date,Home_or_Away,Team,Opponent,Win_Loss,Rest_Days,Points,FG_Made,FG_Attempted,3PT_FG_Made,...,Rebounds,Assists,Steals,Blocks,Turnovers,Season,Home_or_Away_numeric,Team_numeric,Opponent_numeric,Target
Id,,,,,,,,,,,,,,,,,,,,,
1,2023-10-24,Home,GSW,PHX,L,NaN,104,36,101,10,...,49,19,11,6,11,2023-24,1,9,23,0
2,2023-10-24,Away,LAL,DEN,L,NaN,107,41,90,10,...,44,23,5,4,12,2023-24,0,13,7,0
3,2023-10-24,Away,PHX,GSW,W,NaN,108,42,95,11,...,60,23,5,7,19,2023-24,0,23,9,1
4,2023-10-24,Home,DEN,LAL,W,NaN,119,48,91,14,...,42,29,9,6,12,2023-24,1,7,13,1
5,2023-10-25,Away,BOS,NYK,W,NaN,108,37,77,12,...,46,18,6,11,13,2023-24,0,2,19,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7376,2026-04-12,Home,POR,SAC,W,2.0,122,47,101,16,...,46,28,9,3,12,2025-26,1,24,25,1
7377,2026-04-12,Away,SAC,POR,L,2.0,110,40,83,7,...,47,24,7,8,17,2025-26,0,25,24,0
7378,2026-04-12,Home,SAS,DEN,L,2.0,118,45,99,15,...,45,27,5,5,9,2025-26,1,26,7,0


In [58]:
# Train Random Forest model on the first 2 season of the data set (2023-24, 2024-25)
# Returns combined dataframe and precision and accuracy metrics
rf = RandomForestClassifier(n_estimators=50, min_samples_split=10, random_state=1)

def make_predictions(data, predictors):
    train = data[data["Date"] < "2025-10-23"]
    test = data[data["Date"] >= "2025-10-23"]
    rf.fit(train[predictors], train["Target"])
    predictions = rf.predict(test[predictors])

    combined = pd.DataFrame(dict(actual=test["Target"], prediction=predictions), index=test.index) 
    precision = precision_score(test["Target"], predictions)
    accuracy = accuracy_score(test["Target"], predictions) 

    return combined, precision, accuracy

In [59]:
# Function to calculate rolling averages 
def rolling_averages(group, cols, new_cols):
    group = group.sort_values("Date")
    rolling_stats = group[cols].rolling(3 , closed="left").mean() 
    group[new_cols] = rolling_stats 
    group = group.dropna(subset=new_cols) 
    return group

In [60]:
#Compute rolling averages
cols = ["Points","FG_Made","FG_Attempted","3PT_FG_Made","3PT_FG_Attempted","Rebounds","Assists","Steals","Blocks","Turnovers"]
new_cols = [f"{c}_rolling" for c in cols]
matches_rolling = matches.groupby("Team").apply(lambda x: rolling_averages(x, cols, new_cols))
matches_rolling = matches_rolling.reset_index(level='Team')  
matches_rolling.index = range(matches_rolling.shape[0])
matches_rolling

,Team,Date,Home_or_Away,Opponent,Win_Loss,Rest_Days,Points,FG_Made,FG_Attempted,3PT_FG_Made,...,Points_rolling,FG_Made_rolling,FG_Attempted_rolling,3PT_FG_Made_rolling,3PT_FG_Attempted_rolling,Rebounds_rolling,Assists_rolling,Steals_rolling,Blocks_rolling,Turnovers_rolling
0,ATL,2023-10-30,Home,MIN,W,1.0,127,48,86,14,...,119.000000,42.666667,91.000000,10.666667,32.666667,44.000000,28.000000,11.333333,3.000000,14.333333
1,ATL,2023-11-01,Home,WAS,W,2.0,130,46,92,9,...,124.666667,45.666667,88.666667,13.666667,33.000000,42.000000,29.333333,9.333333,5.000000,14.000000
2,ATL,2023-11-04,Away,NOP,W,3.0,123,45,93,14,...,128.000000,47.000000,90.333333,12.666667,33.000000,46.333333,28.666667,9.666667,4.000000,16.333333
3,ATL,2023-11-06,Away,OKC,L,2.0,117,38,102,14,...,126.666667,46.333333,90.333333,12.333333,34.333333,48.333333,27.333333,6.000000,5.333333,15.000000
4,ATL,2023-11-09,Away,ORL,W,3.0,120,41,85,15,...,123.333333,43.000000,95.666667,12.333333,38.333333,56.000000,27.000000,6.666667,4.000000,16.333333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7285,WAS,2026-04-05,Away,BKN,L,1.0,115,44,79,9,...,122.666667,47.000000,98.333333,13.333333,38.333333,36.333333,28.333333,9.000000,3.333333,10.000000
7286,WAS,2026-04-07,Home,CHI,L,2.0,98,37,86,11,...,127.333333,48.000000,93.666667,14.000000,35.000000,39.000000,26.666667,8.333333,3.666667,14.666667
7287,WAS,2026-04-09,Home,CHI,L,2.0,108,38,96,10,...,116.333333,43.666667,90.333333,12.333333,34.333333,43.666667,23.333333,9.666667,5.000000,18.000000
7288,WAS,2026-04-10,Home,MIA,L,1.0,117,46,92,14,...,107.000000,39.666667,87.000000,10.000000,33.666667,46.333333,22.000000,9.666667,6.000000,21.333333


In [61]:
predictors = ["Home_or_Away_numeric", "Opponent_numeric", "Rest_Days"]
combined, precision, accuracy = make_predictions(matches, predictors)
rolling_combined, rolling_precision, rolling_accuracy = make_predictions(matches_rolling, predictors + new_cols)
precision, accuracy, rolling_precision, rolling_accuracy

(0.5502606105733433,
 0.5555098684210527,
 0.5607940446650124,
 0.5604440789473685)